# Python. Семинар 6. Классы вблизи

## 1. `__str__` и `__repr__` — кто из них когда

В лекции оба метода задавали вместе. Разберём, кто их вызывает.

`print` и f-строка берут `__str__`, а вот **контейнер печатает элементы через
`__repr__`** — даже если у класса есть `__str__`.

In [ ]:
class Named:
    def __repr__(self) -> str:
        return "Named(repr)"

    def __str__(self) -> str:
        return "строка"


n = Named()
print(n)
print([n])

Отсюда правило, которого стоит держаться: `__repr__` пишут для того, кто
отлаживает код, — в нём видно тип и содержимое; `__str__` для того, кто читает
вывод программы.

Второй несимметричный момент: если задан только `__repr__`, он подменяет собой и
`__str__`. Обратное неверно — один `__str__` не поможет ни списку, ни отладчику.

In [ ]:
class Only:
    def __repr__(self) -> str:
        return "Only(repr)"


print(Only())
print([Only()])

Поэтому если писать только один метод, писать нужно `__repr__`.

## 2. Свой `__eq__` отменяет хеш

В лекции 3 разбиралось, что в множество и в ключи словаря годятся только
хешируемые объекты. Обычный класс хешируем — по умолчанию сравнение и хеш идут
по тождеству. Но стоит задать своё равенство, как объект перестаёт быть
хешируемым.

In [ ]:
class Point:
    def __init__(self, x: int) -> None:
        self.x = x

    def __eq__(self, other: object) -> bool:
        return isinstance(other, Point) and self.x == other.x


print(Point.__hash__)

`None` вместо метода — это не случайность: определяя `__eq__`, Python явно
затирает унаследованный `__hash__`. Логика простая — равные объекты обязаны
иметь равные хеши, а раз правило равенства изменилось, старый хеш ему больше не
соответствует. Проверить легко:

In [ ]:
try:
    {Point(1)}
except TypeError as error:
    print(type(error).__name__, error)

Чинится двумя способами. Либо задать `__hash__` руками — по тем же полям, что
участвуют в равенстве:

In [ ]:
class Point:
    def __init__(self, x: int) -> None:
        self.x = x

    def __eq__(self, other: object) -> bool:
        return isinstance(other, Point) and self.x == other.x

    def __hash__(self) -> int:
        return hash(self.x)


print(len({Point(1), Point(1), Point(2)}))

Либо взять `dataclass` с `frozen=True`: он и равенство сгенерирует, и хеш, и
запретит менять поля — иначе хеш поехал бы после первого же присваивания.

## 3. `__slots__` перестают работать у наследника

Слоты убирают у объекта словарь атрибутов — отсюда экономия памяти, показанная в
лекции. Но экономия держится ровно до первого наследника, который про слоты
забыл.

In [ ]:
class Base:
    __slots__ = ("x",)


class Child(Base):
    pass


child = Child()
child.что_угодно = 1
print(hasattr(child, "__dict__"), child.что_угодно)

Словарь вернулся, и вместе с ним вернулась возможность дописывать любые
атрибуты — то есть обе выгоды слотов исчезли. Причина в том, что `__slots__`
описывает **только свой** класс; наследник без объявления получает словарь по
умолчанию.

Чтобы цепочка не рвалась, слоты объявляют на каждом уровне — пустым кортежем,
если новых полей нет:

In [ ]:
class Strict(Base):
    __slots__ = ()


strict = Strict()
print(hasattr(strict, "__dict__"))

## 4. Элемент `Enum` не равен своему значению

В лекции перечисления сравнивали между собой. На практике данные приходят
строками — из файла, из ответа сервиса, из аргументов командной строки, — и тут
подстерегает ловушка.

In [ ]:
from enum import Enum


class Device(Enum):
    MAC = "macos"
    WIN = "windows"


print(Device.MAC == "macos")
print(Device.MAC.value == "macos")

Элемент перечисления — самостоятельный объект, а не строка, поэтому прямое
сравнение всегда ложно и никакой ошибки при этом не возникает: условие просто
молча не срабатывает. Сравнивать нужно либо значения, либо — что чище —
превращать строку в элемент перечисления сразу на входе:

In [ ]:
print(Device("macos"), Device["WIN"])

Круглые скобки ищут по значению, квадратные — по имени.

Ещё одна деталь: два имени с одним значением не создают двух элементов. Второе
имя становится псевдонимом первого и в переборе не появляется.

In [ ]:
class Platform(Enum):
    MAC = "macos"
    APPLE = "macos"


print(Platform.APPLE is Platform.MAC)
print([p.name for p in Platform])

## 5. `dataclass` не даёт выстрелить себе в ногу

В лекции 5 разбиралась ловушка изменяемого значения по умолчанию: список,
записанный в умолчание функции, создаётся один раз и живёт между вызовами. В
классе такой список стал бы общим для всех объектов. `dataclass` про это знает и
просто отказывается создавать класс.

In [ ]:
from dataclasses import dataclass

try:

    @dataclass
    class Basket:
        items: list[str] = []

except ValueError as error:
    print(type(error).__name__)
    print(error)

Обратите внимание, когда происходит ошибка: не при создании объекта, а при
**объявлении класса**, то есть сразу при импорте модуля. Ошибку невозможно не
заметить.

Из текста ошибки видно и решение: значение по умолчанию нужно не записывать, а
**вычислять на каждый объект**. Инструмент для этого — `field(default_factory=…)`
из того же модуля; подробно он разбирается в лекции 10 вместе с остальными
фабриками.